<a href="https://colab.research.google.com/github/lndrf-ops/traffic-fine-prediction/blob/main/Process_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Setup

In [3]:
# Installation von pm4py direkt in eurer Colab-Umgebung!pip install pm4py

In [4]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
import os
import zipfile
import pm4py
import pandas as pd

# 1. Falls ihr ein ZIP hochgeladen habt, entpacken wir es zuerst
zip_files = [f for f in os.listdir('.') if f.endswith('.zip')]

if zip_files:
    print(f"ZIP-Datei gefunden: {zip_files[0]}. Entpacke...")
    with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Entpacken abgeschlossen!")

# 2. Sucht nach der entpackten oder direkt hochgeladenen .xes / .xes.gz Datei
xes_files = [f for f in os.listdir('.') if f.endswith('.xes') or f.endswith('.xes.gz')]

if xes_files:
    xes_filename = xes_files[0]
    print(f"Gefundene Datei: '{xes_filename}'. Lese Event Log ein...")

    # Datei mit pm4py einlesen
    event_log = pm4py.read_xes(xes_filename)

    # In Pandas DataFrame konvertieren
    df = pm4py.convert_to_dataframe(event_log)
    print(f"\n🎉 Erfolg! Der Datensatz wurde geladen und hat {len(df)} Events.")
else:
    print("❌ Keine .xes- oder .xes.gz-Datei gefunden. Bitte stellt sicher, dass die Datei hochgeladen wurde und links im Ordner-Menü sichtbar ist.")

In [ ]:
import pandas as pd

print("Starte Analyse und Labeling der Fälle...\n")

# 1. Wir gruppieren die Daten nach der Fall-ID (case:concept:name)
# und sammeln alle Aktivitäten jedes Falls in einer Liste
cases = df.groupby('case:concept:name')['concept:name'].apply(list).reset_index()

# 2. Wir schreiben eine kleine Funktion, die das Ende des Falls bewertet
def determine_outcome(activity_list):
    # Wir prüfen, ob die Ziel-Aktivitäten im Fallverlauf vorkommen
    if 'Payment' in activity_list:
        return 0  # Klasse 0: Erfolg (Bezahlt)
    elif 'Send for Credit Collection' in activity_list:
        return 1  # Klasse 1: Fehlschlag (Inkasso)
    else:
        return -1 # Offener Fall (wird gleich gelöscht)

# 3. Wir wenden die Funktion auf unsere Fälle an
cases['label'] = cases['concept:name'].apply(determine_outcome)

# 4. Wir filtern alle offenen Fälle (-1) heraus
completed_cases = cases[cases['label'] != -1].copy()

# 5. Ergebnisse anzeigen
print(f"Ursprüngliche Anzahl an Fällen: {len(cases)}")
print(f"Abgeschlossene Fälle (für unser ML-Modell nutzbar): {len(completed_cases)}")
print("\nVerteilung unserer Zielvariable (Y):")
print(completed_cases['label'].value_counts().rename(index={0: '0 (Payment)', 1: '1 (Credit Collection)'}))

In [ ]:
print("Starte Feature Engineering für Prefix-Länge k=2...\n")

# 1. Nur die "abgeschlossenen" Fälle im Haupt-Dataframe behalten
valid_cases = completed_cases[['case:concept:name', 'label']]
df_filtered = df.merge(valid_cases, on='case:concept:name', how='inner')

# 2. Chronologisch sortieren, um die Reihenfolge der Events zu garantieren
df_filtered = df_filtered.sort_values(by=['case:concept:name', 'time:timestamp'])

# 3. Position jedes Events im Fall berechnen (1. Event, 2. Event, ...)
df_filtered['event_position'] = df_filtered.groupby('case:concept:name').cumcount() + 1

# 4. Präfixe schneiden: Wir betrachten nur die ersten 2 Events jedes Falls
k = 2
prefixes = df_filtered[df_filtered['event_position'] <= k]

# 5. Wir behalten nur Fälle, die auch wirklich mindestens 2 Events lang wurden
cases_with_min_k_events = prefixes.groupby('case:concept:name').size()
valid_prefix_cases = cases_with_min_k_events[cases_with_min_k_events == k].index
prefixes = prefixes[prefixes['case:concept:name'].isin(valid_prefix_cases)]

# 6. Feature-Tabelle (X) aufbauen
# Feature A: Wie hoch ist die Strafe? (Wir nehmen den Maximalwert des 'amount' aus den ersten 2 Events)
# Hinweis: pm4py lädt oft alles als String, daher wandeln wir 'amount' in Zahlen um
prefixes['amount'] = pd.to_numeric(prefixes['amount'], errors='coerce').fillna(0)
X_amount = prefixes.groupby('case:concept:name')['amount'].max().reset_index()

# Feature B: Welche Aktivitäten sind in den ersten 2 Schritten passiert? (One-Hot-Encoding)
# Das zählt, wie oft "Send Fine", "Add Penalty" etc. in den ersten 2 Schritten vorkamen
X_activities = pd.crosstab(prefixes['case:concept:name'], prefixes['concept:name']).reset_index()

# Alles zusammenfügen
X = X_amount.merge(X_activities, on='case:concept:name')
y = valid_cases[valid_cases['case:concept:name'].isin(X['case:concept:name'])]

# Sortieren, damit X und y exakt übereinstimmen
X = X.sort_values('case:concept:name').drop(columns=['case:concept:name'])
y = y.sort_values('case:concept:name')['label']

print(f"Feature-Matrix X fertiggestellt! Shape: {X.shape}")
print("Wir haben jetzt numerische Features für das Modell.")
display(X.head())

Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd

print("Starte Machine Learning Baseline (Random Forest)...\n")

# 1. Train-Test-Split (80% Training, 20% Test)
# Durch 'stratify=y' stellen wir sicher, dass das Verhältnis von Payment/Inkasso in Train und Test gleich bleibt
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Trainingsdaten: {X_train.shape[0]} Fälle")
print(f"Testdaten: {X_test.shape[0]} Fälle\n")

# 2. Modell initialisieren und trainieren
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
print("Trainiere Random Forest Modell (das dauert ein paar Sekunden)...")
rf_model.fit(X_train, y_train)
print("Training abgeschlossen!\n")

# 3. Vorhersagen auf den Testdaten machen
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1] # Wahrscheinlichkeiten für die ROC-AUC Kurve

# 4. Ergebnisse auswerten
print("--- Klassifikationsbericht ---")
print(classification_report(y_test, y_pred, target_names=['0 (Payment)', '1 (Credit Collection)']))

auc_score = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {auc_score:.4f}\n")

# 5. Feature Importance anzeigen (Warum entscheidet das Modell so?)
importance = pd.DataFrame({'Feature': X.columns, 'Wichtigkeit': rf_model.feature_importances_})
importance = importance.sort_values(by='Wichtigkeit', ascending=False)
print("--- Wichtigste Entscheidungsmerkmale (Feature Importance) ---")
display(importance)

# Training LSTM

In [ ]:
import torch
import numpy as np
from torch.nn.utils.rnn import pad_sequence

print("Bereite Daten für das Deep Learning Modell (LSTM) vor...\n")

# 1. Nur die abgeschlossenen Fälle nutzen
valid_cases_df = df[df['case:concept:name'].isin(completed_cases['case:concept:name'])].copy()
valid_cases_df = valid_cases_df.sort_values(by=['case:concept:name', 'time:timestamp'])

# 2. Vokabular erstellen (jede Aktivität bekommt eine eindeutige ID)
activities = valid_cases_df['concept:name'].unique()
# WICHTIG: ID 0 reservieren wir für das "Padding" (Auffüllen leerer Stellen)
activity_to_id = {act: i+1 for i, act in enumerate(activities)}
num_activities = len(activity_to_id) + 1

print(f"Unser Vokabular umfasst {num_activities-1} Aktivitäten (+1 für Padding-Zero).")

# 3. Aktivitäten durch ihre IDs ersetzen
valid_cases_df['act_id'] = valid_cases_df['concept:name'].map(activity_to_id)

# 4. Präfixe schneiden (Wir schauen uns maximal die ersten 5 Schritte an)
k = 5
valid_cases_df['position'] = valid_cases_df.groupby('case:concept:name').cumcount() + 1
df_prefixes = valid_cases_df[valid_cases_df['position'] <= k]

# 5. Sequenzen als Listen pro Fall zusammenfassen
sequences_df = df_prefixes.groupby('case:concept:name')['act_id'].apply(list).reset_index()
sequences_df = sequences_df.merge(completed_cases[['case:concept:name', 'label']], on='case:concept:name')

# 6. Padding: Alle Sequenzen auf die gleiche Länge (k=5) bringen
# Wir wandeln die Listen in PyTorch-Tensoren um
tensor_sequences = [torch.tensor(seq) for seq in sequences_df['act_id']]
# pad_sequence füllt automatisch mit Nullen auf (padding_value=0)
X_seq = pad_sequence(tensor_sequences, batch_first=True, padding_value=0)
y_seq = torch.tensor(sequences_df['label'].values, dtype=torch.float32)

print(f"\nFertig! Shape unserer Input-Matrix (X_seq): {X_seq.shape}")
print(f"Shape unserer Target-Matrix (y_seq): {y_seq.shape}")

print("\nSo sieht die allererste Sequenz für das Netzwerk aus:")
print(X_seq[0].numpy())

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split

print("Erstelle PyTorch DataLoaders und definiere das Modell...\n")

# 1. Dataset erstellen und in Train/Test aufteilen
dataset = TensorDataset(X_seq, y_seq)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# 2. DataLoader erstellen (Wir trainieren in 64er-Häppchen)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 3. Die LSTM Architektur definieren
class ProcessLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(ProcessLSTM, self).__init__()

        # Der Embedding-Layer wandelt die IDs (1, 2, 3...) in dichte Vektoren um.
        # padding_idx=0 sagt dem Modell, dass ID 0 ("Padding") ignoriert werden soll.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # Das eigentliche LSTM, das die Sequenz lernt
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

        # Ein letzter linearer Layer, der die finale Entscheidung trifft (0 oder 1)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # 1. Sequenz einbetten
        embedded = self.embedding(x)

        # 2. Durch das LSTM schicken
        out, (hn, cn) = self.lstm(embedded)

        # 3. Wir nehmen nur den Output des LETZTEN Zeitschritts (nach 5 Events)
        final_out = out[:, -1, :]

        # 4. Sigmoid wandelt das Ergebnis in eine Wahrscheinlichkeit zwischen 0 und 1 um
        return torch.sigmoid(self.fc(final_out)).squeeze()

# 4. Modell initialisieren
# vocab_size ist 12 (11 Aktivitäten + 1 für die Padding-0)
modell_lstm = ProcessLSTM(vocab_size=12, embedding_dim=16, hidden_dim=32)

print("Modell-Architektur:")
print(modell_lstm)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

print("Starte das Training des LSTM-Modells...\n")

# 1. Loss-Funktion (Binary Cross Entropy für 0/1 Klassifikation) und Optimizer (Adam)
criterion = nn.BCELoss()
optimizer = optim.Adam(modell_lstm.parameters(), lr=0.005)

# 2. Trainings-Schleife (Wir trainieren für 5 Epochen)
epochs = 5
for epoch in range(epochs):
    modell_lstm.train() # Setzt das Modell in den Trainingsmodus
    total_loss = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()           # Alte Berechnungen löschen
        predictions = modell_lstm(batch_X) # Vorhersage machen
        loss = criterion(predictions, batch_y) # Fehler berechnen
        loss.backward()                 # Fehler zurückverfolgen (Backpropagation)
        optimizer.step()                # Gewichte anpassen
        total_loss += loss.item()

    print(f"Epoche {epoch+1}/{epochs} | Durchschnittlicher Fehler (Loss): {total_loss/len(train_loader):.4f}")

print("\nTraining abgeschlossen! Evaluiere Modell auf Testdaten...")

# 3. Evaluation auf den Testdaten
modell_lstm.eval() # Setzt das Modell in den Evaluierungsmodus (kein Lernen mehr)
all_preds = []
all_targets = []

with torch.no_grad(): # Speichert Speicherplatz, da wir keine Gradienten mehr brauchen
    for batch_X, batch_y in test_loader:
        preds = modell_lstm(batch_X)
        all_preds.extend(preds.numpy())
        all_targets.extend(batch_y.numpy())

# Vorhersagen runden (alles > 0.5 wird als Klasse 1 interpretiert)
rounded_preds = [1 if p >= 0.5 else 0 for p in all_preds]

# 4. Ergebnisse ausgeben
print("\n--- Deep Learning (LSTM) Klassifikationsbericht ---")
print(classification_report(all_targets, rounded_preds, target_names=['0 (Payment)', '1 (Credit Collection)']))

auc_score_dl = roc_auc_score(all_targets, all_preds)
print(f"LSTM ROC-AUC Score: {auc_score_dl:.4f}")

# Schließen vom Data Leakage

In [ ]:
import torch
import numpy as np
from torch.nn.utils.rnn import pad_sequence

print("Behebe das Data Leakage: Filtere auf echt laufende Fälle bei k=5...\n")

k = 5

# 1. Wir berechnen die Gesamtlänge jedes abgeschlossenen Falls
case_lengths = valid_cases_df.groupby('case:concept:name').size().reset_index(name='total_length')

# 2. DAS WICHTIGSTE: Wir behalten NUR Fälle, die strikt LÄNGER als k sind!
# So garantieren wir, dass das finale Event (Payment/Inkasso) NICHT in unseren ersten k Schritten auftaucht.
valid_ongoing_cases = case_lengths[case_lengths['total_length'] > k]['case:concept:name']

# 3. Wir filtern unser DataFrame auf diese wirklich laufenden Fälle
df_filtered_leakage = valid_cases_df[valid_cases_df['case:concept:name'].isin(valid_ongoing_cases)].copy()

# 4. Jetzt schneiden wir wieder exakt nach k Schritten ab
df_prefixes_clean = df_filtered_leakage[df_filtered_leakage['position'] <= k]

# 5. Sequenzen bauen
sequences_clean_df = df_prefixes_clean.groupby('case:concept:name')['act_id'].apply(list).reset_index()
sequences_clean_df = sequences_clean_df.merge(completed_cases[['case:concept:name', 'label']], on='case:concept:name')

# 6. Tensoren für PyTorch erstellen
tensor_sequences_clean = [torch.tensor(seq) for seq in sequences_clean_df['act_id']]
X_seq_clean = pad_sequence(tensor_sequences_clean, batch_first=True, padding_value=0)
y_seq_clean = torch.tensor(sequences_clean_df['label'].values, dtype=torch.float32)

print(f"Ursprüngliche Fälle: {len(case_lengths)}")
print(f"Übrig gebliebene, 'echt offene' Fälle für k=5: {len(valid_ongoing_cases)}")
print(f"Shape unserer NEUEN Input-Matrix: {X_seq_clean.shape}")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, roc_auc_score

print("Trainiere LSTM auf den bereinigten (leak-free) Daten...\n")

# 1. DataLoaders mit den NEUEN Daten (X_seq_clean, y_seq_clean) aufbauen
dataset_clean = TensorDataset(X_seq_clean, y_seq_clean)
train_size = int(0.8 * len(dataset_clean))
test_size = len(dataset_clean) - train_size
train_dataset, test_dataset = random_split(dataset_clean, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 2. Modell komplett neu initialisieren (damit es nichts von den alten Daten weiß)
modell_lstm_clean = ProcessLSTM(vocab_size=12, embedding_dim=16, hidden_dim=32)
criterion = nn.BCELoss()
optimizer = optim.Adam(modell_lstm_clean.parameters(), lr=0.005)

# 3. Training
epochs = 5
for epoch in range(epochs):
    modell_lstm_clean.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = modell_lstm_clean(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoche {epoch+1}/5 | Loss: {total_loss/len(train_loader):.4f}")

# 4. Evaluation
print("\nEvaluation auf Testdaten...")
modell_lstm_clean.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        preds = modell_lstm_clean(batch_X)
        all_preds.extend(preds.numpy())
        all_targets.extend(batch_y.numpy())

rounded_preds = [1 if p >= 0.5 else 0 for p in all_preds]

print("\n--- ECHTER Deep Learning Klassifikationsbericht (Kein Data Leakage) ---")
print(classification_report(all_targets, rounded_preds, target_names=['0 (Payment)', '1 (Credit Collection)']))
print(f"ECHTER LSTM ROC-AUC Score: {roc_auc_score(all_targets, all_preds):.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.metrics import classification_report, roc_auc_score

print("Starte finales LSTM-Training mit Class Weights...\n")

# --- 1. DATEN-SETUP ---
dataset_clean = TensorDataset(X_seq_clean, y_seq_clean)
train_size = int(0.8 * len(dataset_clean))
test_size = len(dataset_clean) - train_size
train_dataset, test_dataset = random_split(dataset_clean, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# --- 2. ANGEPASSTE MODELL-ARCHITEKTUR ---
class ProcessLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(ProcessLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.lstm(embedded)
        # Nur den Output des letzten Zeitschritts nehmen
        final_out = out[:, -1, :]
        # WICHTIG: Das torch.sigmoid() ist hier jetzt entfernt!
        return self.fc(final_out).squeeze()

# --- 3. CLASS WEIGHTS BERECHNEN ---
num_payments = (y_seq_clean == 0).sum().item()
num_collections = (y_seq_clean == 1).sum().item()
weight_ratio = num_payments / num_collections
pos_weight_tensor = torch.tensor([weight_ratio])
print(f"Klassengewicht berechnet: Inkasso-Fälle zählen {weight_ratio:.2f}-mal so stark.\n")

# --- 4. MODELL & LOSS INITIALISIEREN ---
# vocab_size ist 12 (11 Aktivitäten + 1 für die Padding-0)
modell_lstm_final = ProcessLSTM(vocab_size=12, embedding_dim=16, hidden_dim=32)

# Neue Loss-Funktion mit der Gewichtung
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = optim.Adam(modell_lstm_final.parameters(), lr=0.005)

# --- 5. TRAININGSSCHLEIFE ---
epochs = 5
for epoch in range(epochs):
    modell_lstm_final.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = modell_lstm_final(batch_X) # Das sind jetzt rohe Logits
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoche {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

# --- 6. EVALUATION ---
print("\nEvaluation auf Testdaten...")
modell_lstm_final.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        # Da das Modell Logits ausgibt, wenden wir für die Metriken das Sigmoid nachträglich an
        logits = modell_lstm_final(batch_X)
        probs = torch.sigmoid(logits)
        all_preds.extend(probs.numpy())
        all_targets.extend(batch_y.numpy())

# Alles über 50% Wahrscheinlichkeit wird als Inkasso (1) gewertet
rounded_preds = [1 if p >= 0.5 else 0 for p in all_preds]

print("\n--- FINALER Deep Learning Klassifikationsbericht ---")
print(classification_report(all_targets, rounded_preds, target_names=['0 (Payment)', '1 (Credit Collection)']))
print(f"FINALER LSTM ROC-AUC Score: {roc_auc_score(all_targets, all_preds):.4f}")

# SHAP Bib integrated

In [ ]:
# 1. SHAP Bibliothek installieren (nur einmalig nötig)
!pip install shap

import shap
import matplotlib.pyplot as plt
import numpy as np

print("Starte SHAP (Explainable AI) für das Random Forest Modell...\n")

# 2. Den SHAP Explainer für unser Random Forest Modell (rf_model) initialisieren
explainer = shap.TreeExplainer(rf_model)

# 3. Um Rechenzeit zu sparen, nehmen wir eine Zufalls-Stichprobe von 1000 Fällen aus den Testdaten
X_test_sample = X_test.sample(1000, random_state=42)

# 4. SHAP-Werte berechnen
shap_values = explainer.shap_values(X_test_sample)

# 5. Visualisierung erstellen (Summary Plot für Klasse 1: Inkasso)
print("Interpretation: Rote Punkte bedeuten einen HOHEN Feature-Wert.")
print("Wenn rote Punkte auf der rechten Seite sind, treibt dieses Feature den Fall ins Inkasso.")

# Bei Random Forests gibt SHAP eine Liste pro Klasse zurück. Index 1 ist Klasse 1 (Inkasso).
plt.title("SHAP Summary Plot: Was treibt einen Fall ins Inkasso?")
shap.summary_plot(shap_values[:, :, 1], X_test_sample, plot_type="dot")

# Conformance Checking

In [ ]:
import pm4py
import pandas as pd

print("Starte Conformance Checking...\n")

# 1. Wir nehmen das originale DataFrame (df), das ihr ganz am Anfang geladen habt
# Wir filtern die 10% häufigsten Varianten, um unser "Soll-Prozessmodell" (Happy Path) zu definieren
happy_path_log = pm4py.filter_variants_top_k(df, 10)

# 2. Wir entdecken das Soll-Modell (Petri-Netz) aus diesen perfekten Daten
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(happy_path_log)
print("Soll-Modell (Petri-Netz) erfolgreich erstellt!")

# 3. Wir nehmen eine Stichprobe von 2000 zufälligen Fällen aus der harten Realität (dem ganzen Log)
# (Eine Stichprobe ist wichtig, da Conformance Checking bei 150k Fällen extrem lange dauert)
sample_cases = df['case:concept:name'].drop_duplicates().sample(2000, random_state=42)
real_world_sample = df[df['case:concept:name'].isin(sample_cases)]

# 4. Fitness berechnen (Token-based Replay)
print("Prüfe die Realität gegen das Soll-Modell (das dauert ein paar Sekunden)...")
fitness = pm4py.fitness_token_based_replay(real_world_sample, net, initial_marking, final_marking)

print("\n--- ERGEBNISSE CONFORMANCE CHECKING ---")
print(f"Durchschnittliche Modell-Fitness: {fitness['log_fitness']*100:.2f} %")
print(f"Prozent der Fälle, die den Prozess PERFEKT ohne Abweichung durchlaufen: {fitness['perc_fit_traces']:.2f} %")

# Systematic Comparison


In [ ]:
import pandas as pd

print("Erstelle systematischen Modell-Vergleich...\n")

# Wir fassen die Metriken aus euren bisherigen Outputs zusammen
comparison_data = {
    'Modell-Iteration': [
        '1. Random Forest Baseline (k=2)',
        '2. LSTM (k=5) - Naive',
        '3. LSTM (k=5) - Final'
    ],
    'Status / Pain Point': [
        'Data Leakage ("Payment" im Prefix)',
        'Extremes Leakage (100% Accuracy)',
        'Leakage gefixt + Class Weights'
    ],
    'Accuracy': ['82 %', '100 %', '92 %'],
    'Recall (Inkasso)': ['100 %', '99 %', '100 %'],
    'Precision (Inkasso)': ['72 %', '100 %', '35 %'],
    'ROC-AUC Score': ['0.865', '1.000', '0.956']
}

# DataFrame erstellen
df_comparison = pd.DataFrame(comparison_data)

# Schöne Ausgabe
display(df_comparison.style.set_properties(**{'text-align': 'left', 'background-color': '#f4f4f4'})
        .set_table_styles([dict(selector='th', props=[('text-align', 'left'), ('background-color', '#d9ead3')])])
        .hide(axis="index"))

# Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("Erstelle systematischen Modell-Vergleich...\n")

comparison_data = {
    'Modell-Iteration': [
        '1. Random Forest\n(Baseline k=2)',
        '2. LSTM Naive\n(Data Leakage k=5)',
        '3. LSTM Final\n(Gefixt + Class Weights)'
    ],
    'Accuracy': [0.82, 1.00, 0.92],
    'Recall (Inkasso)': [1.00, 0.99, 1.00],
    'Precision (Inkasso)': [0.72, 1.00, 0.35],
    'ROC-AUC Score': [0.865, 1.000, 0.956]
}

df_comparison = pd.DataFrame(comparison_data)

# 1. TABELLE ANZEIGEN (Fix für Dark Mode: Keine Hintergrundfarben erzwingen!)
display(df_comparison.style.format({
    'Accuracy': "{:.0%}",
    'Recall (Inkasso)': "{:.0%}",
    'Precision (Inkasso)': "{:.0%}",
    'ROC-AUC Score': "{:.3f}"
}).hide(axis="index"))

print("\nGeneriere Diagramm für die Präsentation...\n")

# 2. DIAGRAMM FÜR DIE SLIDES GENERIEREN
# Setup
labels = df_comparison['Modell-Iteration']
x = np.arange(len(labels))  # Positionen der Labels
width = 0.2  # Breite der Balken

fig, ax = plt.subplots(figsize=(10, 6))

# Balken zeichnen
rects1 = ax.bar(x - 1.5*width, df_comparison['Accuracy'], width, label='Accuracy')
rects2 = ax.bar(x - 0.5*width, df_comparison['Recall (Inkasso)'], width, label='Recall (Inkasso)')
rects3 = ax.bar(x + 0.5*width, df_comparison['Precision (Inkasso)'], width, label='Precision (Inkasso)')
rects4 = ax.bar(x + 1.5*width, df_comparison['ROC-AUC Score'], width, label='ROC-AUC Score')

# Text, Titel und Achsen
ax.set_ylabel('Scores (0.0 bis 1.0)')
ax.set_title('Systematischer Modellvergleich (Predictive Analytics)', pad=20, fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.25), ncol=4)
ax.set_ylim(0, 1.1)

# Werte über den Balken anzeigen
ax.bar_label(rects1, padding=3, fmt='%.2f')
ax.bar_label(rects2, padding=3, fmt='%.2f')
ax.bar_label(rects3, padding=3, fmt='%.2f')
ax.bar_label(rects4, padding=3, fmt='%.2f')

fig.tight_layout()
plt.show()

# Saving Model


In [ ]:
import joblib

# 1. Wir speichern das trainierte Random Forest Modell
joblib.dump(rf_model, 'rf_model.pkl')

# 2. Wir speichern die exakten Feature-Namen (Aktivitäten), die das Modell erwartet
joblib.dump(list(X.columns), 'model_features.pkl')

print("Modell und Features erfolgreich als Datei gespeichert!")